In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "cross-encoder/nli-MiniLM2-L6-H768"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()} if isinstance(model.config.id2label, dict) else model.config.id2label
label2id = {str(v).lower(): int(k) for k, v in id2label.items()}

if "entailment" not in label2id:
    raise ValueError(f"Expected an entailment label in model labels, found: {id2label}")

entailment_id = label2id["entailment"]
print(f"Loaded model: {model_name}")
print(f"Model labels: {id2label}")
print(f"Using fixed mapping: entailment label id = {entailment_id} -> paraphrase prediction when argmax == entailment")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
batch_size = 32
labels = dataset["label"]
predictions = []
positive_scores = []
predicted_label_names = []
pair_lengths = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    pair_lengths.extend([int(x) for x in inputs["attention_mask"].sum(dim=1).tolist()])
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(logits, dim=-1)
    predictions.extend((pred_ids == entailment_id).long().cpu().tolist())
    positive_scores.extend(probs[:, entailment_id].cpu().tolist())
    predicted_label_names.extend([id2label[int(i)] for i in pred_ids.cpu().tolist()])

print(f"Completed inference for {len(predictions)} examples.")
print(f"Tokenized pair length stats -> min={min(pair_lengths)}, max={max(pair_lengths)}, avg={sum(pair_lengths)/len(pair_lengths):.2f}")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

print("Overall evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
n = len(dataset)
order = sorted(range(n), key=lambda i: positive_scores[i])
decile_size = max(1, n // 10)

bottom_decile_idxs = order[:decile_size]
middle_start = max(0, (n // 2) - (decile_size // 2))
middle_decile_idxs = order[middle_start:middle_start + decile_size]
top_decile_idxs = order[-decile_size:]

score_groups = {
    "bottom_decile": bottom_decile_idxs,
    "middle_decile": middle_decile_idxs,
    "top_decile": top_decile_idxs,
}

def summarize_group(idxs):
    y_true = [labels[i] for i in idxs]
    y_pred = [predictions[i] for i in idxs]
    scores = [positive_scores[i] for i in idxs]
    group_precision, group_recall, group_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "count": len(idxs),
        "positive_rate_true": float(np.mean(y_true)) if len(y_true) else 0.0,
        "positive_rate_pred": float(np.mean(y_pred)) if len(y_pred) else 0.0,
        "precision": group_precision,
        "recall": group_recall,
        "f1": group_f1,
        "score_min": float(min(scores)) if len(scores) else 0.0,
        "score_max": float(max(scores)) if len(scores) else 0.0,
        "score_avg": float(sum(scores) / len(scores)) if len(scores) else 0.0,
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }

group_metrics = {name: summarize_group(idxs) for name, idxs in score_groups.items()}

print("Score-decile breakdowns:")
for name in ["bottom_decile", "middle_decile", "top_decile"]:
    m = group_metrics[name]
    print(
        f"group={name} count={m['count']} score_range=({m['score_min']:.4f}, {m['score_max']:.4f}) "
        f"score_avg={m['score_avg']:.4f} true_positive_rate={m['positive_rate_true']:.4f} "
        f"pred_positive_rate={m['positive_rate_pred']:.4f} precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f}"
    )
    print(f"confusion_matrix={m['confusion_matrix']}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

high_score_false_positives = []
low_score_false_negatives = []

for i in range(n):
    record = {
        "idx": i,
        "score": positive_scores[i],
        "true_label": labels[i],
        "pred_label": predictions[i],
        "predicted_nli_label": predicted_label_names[i],
        "length": pair_lengths[i],
        "sentence1": dataset[i]["sentence1"],
        "sentence2": dataset[i]["sentence2"],
    }
    if labels[i] == 0 and predictions[i] == 1:
        high_score_false_positives.append(record)
    if labels[i] == 1 and predictions[i] == 0:
        low_score_false_negatives.append(record)

high_score_false_positives = sorted(high_score_false_positives, key=lambda x: (-x["score"], -x["length"]))
low_score_false_negatives = sorted(low_score_false_negatives, key=lambda x: (x["score"], -x["length"]))

num_fp_to_show = min(5, len(high_score_false_positives))
num_fn_to_show = min(5, len(low_score_false_negatives))

print(f"High-score false positives: showing {num_fp_to_show} of {len(high_score_false_positives)}")
for item in high_score_false_positives[:num_fp_to_show]:
    print(f"Index: {item['idx']}")
    print(f"tokenized_pair_length: {item['length']}")
    print(f"positive_class_score(entailment): {item['score']:.4f}")
    print(f"sentence1: {item['sentence1']}")
    print(f"sentence2: {item['sentence2']}")
    print(f"true label: {item['true_label']} ({label_map[item['true_label']]})")
    print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
    print(f"predicted_nli_label: {item['predicted_nli_label']}")
    print("-" * 80)

print(f"Low-score false negatives: showing {num_fn_to_show} of {len(low_score_false_negatives)}")
for item in low_score_false_negatives[:num_fn_to_show]:
    print(f"Index: {item['idx']}")
    print(f"tokenized_pair_length: {item['length']}")
    print(f"positive_class_score(entailment): {item['score']:.4f}")
    print(f"sentence1: {item['sentence1']}")
    print(f"sentence2: {item['sentence2']}")
    print(f"true label: {item['true_label']} ({label_map[item['true_label']]})")
    print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
    print(f"predicted_nli_label: {item['predicted_nli_label']}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"entailment_label_id={entailment_id}")
print("prediction_rule=argmax_label_equals_entailment_implies_paraphrase")
print(f"overall_accuracy={accuracy:.4f}")
print(f"overall_precision={precision:.4f}")
print(f"overall_recall={recall:.4f}")
print(f"overall_f1={f1:.4f}")
for name in ["bottom_decile", "middle_decile", "top_decile"]:
    m = group_metrics[name]
    print(f"{name}_count={m['count']}")
    print(f"{name}_score_min={m['score_min']:.4f}")
    print(f"{name}_score_max={m['score_max']:.4f}")
    print(f"{name}_score_avg={m['score_avg']:.4f}")
    print(f"{name}_true_positive_rate={m['positive_rate_true']:.4f}")
    print(f"{name}_pred_positive_rate={m['positive_rate_pred']:.4f}")
    print(f"{name}_precision={m['precision']:.4f}")
    print(f"{name}_recall={m['recall']:.4f}")
    print(f"{name}_f1={m['f1']:.4f}")
print(f"high_score_false_positive_count={len(high_score_false_positives)}")
print(f"low_score_false_negative_count={len(low_score_false_negatives)}")